# 05 — Held-Out Test Set Evaluation (Botobot / Group 6)

**Project:** Mapúa University AI3 Machine Project — Philippine Voting Process Helper NLP (`Botobot / Group 6`)  
**Test set:** `data/eval/test_50_questions.jsonl`: 50 unseen questions (40 answerable incl. 9 Filipino/Taglish, 10 unanswerable abstention traps)  
**Scripts:** `eval/leakage_check.py`, `eval/retrieval_eval.py`, `eval/score_predictions.py` (see `eval/README.md`)  
**Target Runtime:** Google Colab (`L4 24 GB` / `T4 16 GB`)

**Run order**
1. **Part A** (this notebook, Cells 1–6): leakage check, retrieval evaluation, and writing `data/eval/frozen_top5_cache.jsonl`.
2. Run **Cells 1–9 of `03_Run_Qwen35_4B_RAG.ipynb` and `04_Run_Gemma4_E4B_RAG.ipynb`**. Cell 8 loads the frozen cache, so every model answers from the same top-5 contexts.
3. **Part B** (this notebook, Cells 7–9): score every `eval_outputs/*_preds.jsonl` and compare the models.

**Drive layout expected** (same root as notebooks 01–04):
```
AI3_Botobot/
  data/index/   chunks_index.jsonl, bge_m3_embeddings.npy, bm25_tokens.json, retrieval_smoke_test.json
  data/eval/    test_50_questions.jsonl
  data/sft/     qwen35/, gemma4/   (for the leakage check)
  eval/         common.py, leakage_check.py, retrieval_eval.py, score_predictions.py
  eval_outputs/ (created)
```

In [ ]:
# @title Cell 1: Install Dependencies
!pip -q install -U "transformers>=5.5.0" accelerate faiss-cpu rank-bm25

In [ ]:
# @title Cell 2: Mount Google Drive, Paths & File Checks
import json
import os
import sys
from pathlib import Path

USE_DRIVE = os.environ.get("AI3_USE_DRIVE", "1") == "1"
if os.environ.get("AI3_BASE_DIR"):
    BASE_DIR = Path(os.environ["AI3_BASE_DIR"])
elif USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE_DIR = Path("/content/drive/MyDrive/AI3_Botobot")
    except Exception as exc:
        print(f"[WARN] Drive mount unavailable ({exc}); falling back to /content/AI3_Botobot")
        BASE_DIR = Path("/content/AI3_Botobot")
else:
    BASE_DIR = Path("/content/AI3_Botobot")

os.environ["AI3_BASE_DIR"] = str(BASE_DIR)   # read by eval/common.py
os.chdir(BASE_DIR)
sys.path.insert(0, str(BASE_DIR / "eval"))

required = [
    "data/index/chunks_index.jsonl", "data/index/bge_m3_embeddings.npy", "data/index/bm25_tokens.json",
    "data/index/retrieval_smoke_test.json", "data/eval/test_50_questions.jsonl",
    "eval/common.py", "eval/leakage_check.py", "eval/retrieval_eval.py", "eval/score_predictions.py",
]
missing = [p for p in required if not (BASE_DIR / p).exists()]
assert not missing, f"Upload these to {BASE_DIR} first: {missing}"

from common import load_test_set
TEST = load_test_set()
print(f"Base dir: {BASE_DIR}")
print(f"Test set: {len(TEST)} questions | answerable={sum(t['is_answerable'] for t in TEST)} "
      f"| unanswerable={sum(not t['is_answerable'] for t in TEST)} "
      f"| Filipino/Taglish={sum(t['language'] == 'tl' for t in TEST)}")
print("SFT files for leakage check:", sorted(str(p.relative_to(BASE_DIR)) for p in (BASE_DIR / "data" / "sft").glob("*/*.jsonl")))

## Part A: Is the test set unseen, and does retrieval find the right sections?

In [ ]:
# @title Cell 3: Leakage Check (test questions vs. smoke-test queries and SFT train/val prompts)
!python eval/leakage_check.py

report = json.loads((BASE_DIR / "eval_outputs" / "leakage_report.json").read_text())
print("\nVerdict:", report["verdict"], "| dev questions compared:", report["n_dev_questions"])
overlap = report["sft_section_overlap"]
if isinstance(overlap, list):
    print(f"Test questions whose gold section also appears in SFT data: {len(overlap)} / {len(TEST)}")
    for o in overlap:
        print("  ", o["test_id"], o["overlapping_sections"])
else:
    print(overlap)

In [ ]:
# @title Cell 4: Retrieval Evaluation (dense vs BM25 vs hybrid RRF vs hybrid + reranker) + Frozen Top-5 Cache
import torch
assert torch.cuda.is_available(), "Select a GPU runtime (T4 or L4)."
!python eval/retrieval_eval.py

In [ ]:
# @title Cell 5: Retrieval Results Tables
import pandas as pd
from IPython.display import display, Markdown

ret = json.loads((BASE_DIR / "eval_outputs" / "retrieval_eval_summary.json").read_text())
cfg_names = {"dense": "Dense (bge-m3)", "bm25": "BM25", "hybrid": "Hybrid RRF", "hybrid_rr": "Hybrid RRF + reranker"}
ret_df = pd.DataFrame(ret["configs"]).T.rename(index=cfg_names)
display(Markdown(f"**Retrieval on {ret['n_answerable']} answerable held-out questions** ({ret['device']})"))
display(ret_df.style.format("{:.3f}").highlight_max(axis=0, color="#c8e6c9"))

display(Markdown("**Hybrid + reranker by category / language (Hit@5, MRR@20)**"))
display(pd.DataFrame(ret["by_category"]).T)
display(pd.DataFrame(ret["by_language"]).T)

display(Markdown("**Retrieval abstention gate** (`min_dense=0.42`, `min_rerank=-2.0`)"))
display(pd.Series(ret["abstention_gate"]).to_frame("value"))

display(Markdown("**Questions whose gold section was not in the top 5**"))
display(pd.DataFrame(ret["misses_at_5"]) if ret["misses_at_5"] else Markdown("_none_"))

In [ ]:
# @title Cell 6: Retrieval Charts (config comparison + gate score separation)
import matplotlib.pyplot as plt
from common import read_jsonl

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
metrics = ["hit@1", "hit@5", "mrr@20", "ndcg@5"]
ret_df[metrics].plot.bar(ax=axes[0], rot=0, width=0.8)
axes[0].set_ylim(0, 1.05); axes[0].set_title("Retriever configurations (answerable questions)")
axes[0].legend(ncol=4, fontsize=8, loc="lower right"); axes[0].grid(axis="y", alpha=0.3)

per_q = pd.DataFrame(read_jsonl(BASE_DIR / "eval_outputs" / "retrieval_eval_per_question.jsonl"))
for ans, color, label in [(True, "#2e7d32", "answerable"), (False, "#c62828", "unanswerable")]:
    d = per_q[per_q.is_answerable == ans]
    axes[1].scatter(d.max_dense_top5, d.max_rerank_top5, c=color, label=label, alpha=0.75)
axes[1].axvline(0.42, ls="--", c="gray"); axes[1].axhline(-2.0, ls="--", c="gray")
axes[1].set_xlabel("max dense cosine in top-5"); axes[1].set_ylabel("max reranker logit in top-5")
axes[1].set_title("Abstention gate: gate fires only in the lower-left box"); axes[1].legend()
plt.tight_layout()
(BASE_DIR / "eval_outputs" / "figures").mkdir(exist_ok=True)
plt.savefig(BASE_DIR / "eval_outputs" / "figures" / "retrieval_eval.png", dpi=150)
plt.show()
print("Frozen cache ready:", (BASE_DIR / "data" / "eval" / "frozen_top5_cache.jsonl").exists())

## Part B: Generated answers

Run Cells 1–9 of notebooks **03** and **04** first, for every variant you want to compare: base vs. LoRA, 8-bit vs. 16-bit (`AI3_USE_LORA`, `AI3_PRECISION`). Each run writes `eval_outputs/<model>_<precision>_preds.jsonl`.

> Cell 9 names its output `<model>_<precision>_preds.jsonl` and does not include the LoRA flag, so a base run and a LoRA run at the same precision overwrite each other. After each run, rename the file, e.g. `qwen35_8bit_lora_preds.jsonl` / `qwen35_8bit_base_preds.jsonl`.

In [ ]:
# @title Cell 7: Score All Prediction Files
import glob
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown
from common import read_jsonl
(BASE_DIR / "eval_outputs" / "figures").mkdir(parents=True, exist_ok=True)
# Part B can run in a fresh session: re-run Cells 1-2 first, then start here.
pred_files = sorted(glob.glob(str(BASE_DIR / "eval_outputs" / "*_preds.jsonl")))
assert pred_files, "No *_preds.jsonl yet. Run Cells 8-9 of notebooks 03/04 first."
print("Scoring:", [Path(p).name for p in pred_files])
args = " ".join(f'"{p}"' for p in pred_files)
!python eval/score_predictions.py {args} > /dev/null

summaries = {Path(p).stem: json.loads((BASE_DIR / "eval_outputs" / f"{Path(p).stem}_score_summary.json").read_text())
             for p in pred_files}
rows = {}
for name, s in summaries.items():
    rows[name] = {
        "strict_accuracy": s["answerable"]["strict_accuracy"],
        "key_fact_recall": s["answerable"]["key_fact_recall"],
        "token_f1": s["answerable"]["token_f1"],
        "citation_accuracy": s["answerable"]["citation_accuracy"],
        "false_abstention": s["answerable"]["false_abstention_rate"],
        "correct_abstention": s["unanswerable"]["correct_abstention_rate"],
        "number_grounding": s["grounding"]["number_grounding"],
        "sentence_support": s["grounding"]["sentence_support"],
        "acc_gold_in_top5": s["retrieval_effect"]["accuracy_when_gold_in_top5"],
        "acc_gold_missing": s["retrieval_effect"]["accuracy_when_gold_missing"],
        "latency_mean_s": s["latency_s"]["mean"],
        "latency_p95_s": s["latency_s"]["p95"],
    }
gen_df = pd.DataFrame(rows).T
display(Markdown("**Held-out test set: generation metrics per run**"))
display(gen_df.style.format("{:.3f}", na_rep="-"))
gen_df.to_csv(BASE_DIR / "eval_outputs" / "generation_metrics_table.csv")

In [ ]:
# @title Cell 8: Per-Category / Per-Language Accuracy + Charts
cat_df = pd.DataFrame({n: {c: v["strict_accuracy"] for c, v in s["by_category"].items()} for n, s in summaries.items()})
lang_df = pd.DataFrame({n: {l: v["strict_accuracy"] for l, v in s["by_language"].items()} for n, s in summaries.items()})
display(Markdown("**Strict accuracy by category**")); display(cat_df.style.format("{:.2f}", na_rep="-"))
display(Markdown("**Strict accuracy by language (en / tl = Filipino-Taglish)**")); display(lang_df.style.format("{:.2f}", na_rep="-"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
gen_df[["strict_accuracy", "citation_accuracy", "correct_abstention", "number_grounding"]].plot.bar(ax=axes[0], rot=15, width=0.8)
axes[0].set_ylim(0, 1.05); axes[0].set_title("Answer quality per run"); axes[0].grid(axis="y", alpha=0.3); axes[0].legend(fontsize=8)
gen_df[["acc_gold_in_top5", "acc_gold_missing"]].plot.bar(ax=axes[1], rot=15, color=["#2e7d32", "#c62828"])
axes[1].set_ylim(0, 1.05); axes[1].set_title("Effect of retrieval: accuracy when gold section retrieved vs. missed")
axes[1].grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(BASE_DIR / "eval_outputs" / "figures" / "generation_eval.png", dpi=150); plt.show()

In [ ]:
# @title Cell 9: Failure Cases & Sample Q&A (retrieved context vs. generated answer)
import textwrap
from common import load_chunks
sec_of = {c["chunk_id"]: c["section_key"] for c in load_chunks()}
test_by_id = {t["id"]: t for t in TEST}

RUN = list(summaries)[-1]          # change to the run you want to inspect
preds = {p["id"]: p for p in read_jsonl(BASE_DIR / "eval_outputs" / f"{RUN}.jsonl")}
fails = summaries[RUN]["failures"]
print(f"Run: {RUN}")
for k, v in fails.items():
    print(f"  {k}: {v}")

def show(qid):
    t, p = test_by_id[qid], preds[qid]
    print("=" * 100)
    print(f"[{qid}] ({t['category']}, {t['language']}, answerable={t['is_answerable']})  {t['question']}")
    print("Gold sections:", t["gold_section_keys"] or "- (should abstain)")
    print("Retrieved:    ", [sec_of.get(c, c).split("::", 1)[-1] for c in p["retrieved_chunk_ids"]])
    print("Reference:", textwrap.fill(t["ground_truth"], 100, subsequent_indent="           "))
    print("Model:    ", textwrap.fill(p["answer"], 100, subsequent_indent="           "))

# Two correct answers, then every failure
ok = [i for i in preds if i not in sum(fails.values(), [])][:2]
for qid in ok + sorted(set(sum(fails.values(), []))):
    show(qid)